# Support Ticket Env - GRPO Fine-Tuning
**OpenEnv x Scalar Hackathon**

Fine-tunes `Qwen/Qwen2.5-0.5B-Instruct` using GRPO (Group Relative Policy Optimization) from HuggingFace TRL against the live Support Ticket Environment API.

- Model: Qwen2.5-0.5B-Instruct
- Algorithm: GRPO
- Environment: https://algocore-support-ticket-env.hf.space
- Runtime: ~45-60 min on free Colab T4

In [ ]:
!pip install -q trl transformers peft accelerate
!pip install -q torch bitsandbytes requests datasets
print('Done')

In [ ]:
import os

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')
if not HF_TOKEN:
    raise ValueError('Set HF_TOKEN in Colab Secrets: click the key icon in the left panel')

ENV_BASE_URL = 'https://algocore-support-ticket-env.hf.space'
MODEL_NAME   = 'Qwen/Qwen2.5-0.5B-Instruct'
OUTPUT_DIR   = '/content/support-ticket-grpo'
HF_REPO_ID   = 'AlgoCore/support-ticket-grpo-model'

os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU - switch runtime!')
if torch.cuda.is_available():
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
print('Model:', MODEL_NAME)
print('Env:', ENV_BASE_URL)

In [ ]:
import requests
import json
import re
from dataclasses import dataclass
from typing import Optional

@dataclass
class Obs:
    ticket_id: str
    ticket_text: str
    task_id: int
    current_category: Optional[str]
    resolved: bool
    step_count: int
    feedback: str
    score: float
    reward: float
    done: bool

class SupportEnvClient:
    def __init__(self, base_url):
        self.base_url = base_url.rstrip('/')
        self.session = requests.Session()
        self.session.headers.update({'Content-Type': 'application/json'})

    def health(self):
        try:
            r = self.session.get(f"{self.base_url}/health", timeout=10)
            return r.status_code == 200
        except:
            return False

    def reset(self, task_id=1, seed=42):
        r = self.session.post(f"{self.base_url}/reset", json={"task_id": task_id, "seed": seed}, timeout=15)
        r.raise_for_status()
        return self._parse(r.json())

    def step(self, action):
        r = self.session.post(f"{self.base_url}/step", json={"action": action}, timeout=15)
        r.raise_for_status()
        return self._parse(r.json())

    def _parse(self, data):
        obs = data.get('observation', data)
        return Obs(
            ticket_id=obs.get('ticket_id', ''),
            ticket_text=obs.get('ticket_text', ''),
            task_id=obs.get('task_id', 1),
            current_category=obs.get('current_category'),
            resolved=obs.get('resolved', False),
            step_count=obs.get('step_count', 0),
            feedback=obs.get('feedback', ''),
            score=obs.get('score', 0.0),
            reward=obs.get('reward', 0.0),
            done=obs.get('done', False),
        )

env_client = SupportEnvClient(ENV_BASE_URL)
if env_client.health():
    print('Environment API reachable')
    obs = env_client.reset(task_id=1, seed=42)
    print(f'Ticket: {obs.ticket_id} - {obs.ticket_text[:70]}')
else:
    print('Cannot reach environment - check ENV_BASE_URL')

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

print(f"Loading {MODEL_NAME}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    torch_dtype=torch.float16,
    device_map='auto',
    trust_remote_code=True,
)

print(f'Model loaded - {sum(p.numel() for p in model.parameters())/1e6:.0f}M parameters')
print(f'Device: {next(model.parameters()).device}')

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
SYSTEM_PROMPT = """You are a customer support AI agent. Given a ticket, respond with a JSON action.

Respond ONLY with valid JSON:
{"action_type": "classify"|"reply"|"escalate"|"close", "category": "billing"|"technical"|"account"|"general"|"refund", "reply_text": "...", "reason": "..."}

Rules:
- Task 1: action_type=classify, pick correct category
- Task 2: first classify, then reply/escalate/close
- Task 3: classify each ticket then resolve it
- category only needed for classify
- reply_text only needed for reply
- technical issues: escalate
- resolved issues: close
- billing/account/refund: reply"""

def build_prompt(obs):
    user_msg = json.dumps({
        "ticket_id": obs.ticket_id,
        "ticket_text": obs.ticket_text,
        "task_id": obs.task_id,
        "current_category": obs.current_category,
        "feedback": obs.feedback,
        "step_count": obs.step_count,
    }, indent=2)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_msg},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def parse_action(text):
    text = text.strip()
    text = re.sub(r'^```(?:json)?\s*', '', text)
    text = re.sub(r'\s*```$', '', text)
    try:
        return json.loads(text)
    except:
        match = re.search(r'\{.*?\}', text, re.DOTALL)
        if match:
            try:
                return json.loads(match.group())
            except:
                pass
    return {"action_type": "classify", "category": "general"}

obs = env_client.reset(task_id=1, seed=42)
prompt = build_prompt(obs)
print('Prompt builder OK')
print(f'Prompt length: {len(prompt)} chars')

In [ ]:
import random

SEEDS    = [42, 7, 123, 0, 99]
TASK_IDS = [1, 2, 3]
MAX_STEPS = 6

def generate_action(prompt, max_new_tokens=150):
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=1024).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

def run_episode(task_id, seed):
    obs = env_client.reset(task_id=task_id, seed=seed)
    prompts, completions, rewards = [], [], []
    for _ in range(MAX_STEPS):
        if obs.done:
            break
        prompt = build_prompt(obs)
        completion = generate_action(prompt)
        action = parse_action(completion)
        try:
            obs = env_client.step(action)
            reward = float(obs.reward or 0.0)
        except:
            reward = -0.1
            obs.done = True
        prompts.append(prompt)
        completions.append(completion)
        rewards.append(reward)
        if obs.done:
            break
    return prompts, completions, sum(rewards)

print('Running smoke test...')
p, c, r = run_episode(task_id=1, seed=42)
print(f'Smoke test passed - steps={len(p)}, total_reward={r:.3f}')

In [ ]:
def evaluate(n_seeds=3):
    results = {}
    seeds = SEEDS[:n_seeds]
    for task_id in [1, 2, 3]:
        task_rewards = []
        for seed in seeds:
            _, _, total = run_episode(task_id=task_id, seed=seed)
            normalized = round(max(0, min(1, total / MAX_STEPS)), 3)
            task_rewards.append(normalized)
        avg = round(sum(task_rewards) / len(task_rewards), 3)
        results[f'task{task_id}'] = avg
        print(f'  Task {task_id}: {avg:.3f}')
    results['overall'] = round(sum(results.values()) / 3, 3)
    print(f'  Overall: {results["overall"]:.3f}')
    return results

print('=== BASELINE (before training) ===')
baseline_scores = evaluate(n_seeds=3)

In [ ]:
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
import numpy as np

LEARNING_RATE = 5e-5
N_EPISODES    = 60
GROUP_SIZE    = 4
KL_COEFF      = 0.01
GRAD_CLIP     = 1.0
LOG_EVERY     = 5

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=5, num_training_steps=N_EPISODES)

training_log = []

print(f'Starting GRPO training: {N_EPISODES} episodes, group_size={GROUP_SIZE}')
print('=' * 60)

model.train()

for episode in range(1, N_EPISODES + 1):
    task_id = random.choice(TASK_IDS)
    seed    = random.choice(SEEDS)

    group_rewards     = []
    group_prompts     = []
    group_completions = []

    for g in range(GROUP_SIZE):
        obs = env_client.reset(task_id=task_id, seed=seed)
        prompt = build_prompt(obs)
        completion = generate_action(prompt)
        action = parse_action(completion)
        try:
            obs = env_client.step(action)
            reward = float(obs.reward or 0.0)
        except:
            reward = -0.1
        group_rewards.append(reward)
        group_prompts.append(prompt)
        group_completions.append(completion)

    rewards_arr = np.array(group_rewards, dtype=np.float32)
    advantages  = (rewards_arr - rewards_arr.mean()) / (rewards_arr.std() + 1e-8)

    total_loss = torch.tensor(0.0, requires_grad=True, device=model.device)
    optimizer.zero_grad()

    for prompt, completion, adv in zip(group_prompts, group_completions, advantages):
        if not completion.strip():
            continue
        full_text = prompt + completion
        inputs = tokenizer(full_text, return_tensors='pt', truncation=True, max_length=1200).to(model.device)
        prompt_len = tokenizer(prompt, return_tensors='pt')["input_ids"].shape[1]
        outputs = model(**inputs, labels=inputs['input_ids'])
        logits = outputs.logits[:, prompt_len-1:-1, :]
        target_ids = inputs['input_ids'][:, prompt_len:]
        if target_ids.shape[1] == 0:
            continue
        log_probs = torch.nn.functional.log_softmax(logits, dim=-1)
        token_log_probs = log_probs.gather(2, target_ids.unsqueeze(-1)).squeeze(-1)
        seq_log_prob = token_log_probs.mean()
        pg_loss  = -torch.tensor(float(adv), device=model.device) * seq_log_prob
        kl_loss  = KL_COEFF * (seq_log_prob ** 2)
        total_loss = total_loss + (pg_loss + kl_loss) / GROUP_SIZE

    total_loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
    optimizer.step()
    scheduler.step()

    avg_reward = float(rewards_arr.mean())
    training_log.append((episode, task_id, avg_reward))

    if episode % LOG_EVERY == 0:
        print(f'Episode {episode:3d}/{N_EPISODES} | task={task_id} | avg_reward={avg_reward:.3f} | loss={total_loss.item():.4f}')

print('Training complete!')

In [ ]:
model.eval()

print('=== POST-TRAINING EVALUATION ===')
trained_scores = evaluate(n_seeds=3)

print('\n=== IMPROVEMENT SUMMARY ===')
print(f'{"Task":<10} {"Before":>8} {"After":>8} {"Delta":>8}')
print('-' * 38)
for key, label in [("task1","Task 1"),("task2","Task 2"),("task3","Task 3"),("overall","Overall")]:
    b = baseline_scores.get(key, 0)
    a = trained_scores.get(key, 0)
    d = a - b
    print(f'{label:<10} {b:>8.3f} {a:>8.3f} {d:>+8.3f}')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

episodes   = [x[0] for x in training_log]
task_ids   = [x[1] for x in training_log]
ep_rewards = [x[2] for x in training_log]

def moving_avg(data, window=5):
    return np.convolve(data, np.ones(window)/window, mode='valid')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Support Ticket Env - GRPO Training Results', fontsize=14, fontweight='bold')

ax1 = axes[0]
colors = {1: '#3498db', 2: '#2ecc71', 3: '#e74c3c'}
for tid in [1, 2, 3]:
    mask = [i for i, t in enumerate(task_ids) if t == tid]
    if mask:
        x = [episodes[i] for i in mask]
        y = [ep_rewards[i] for i in mask]
        ax1.scatter(x, y, alpha=0.3, color=colors[tid], s=15)
        if len(y) >= 5:
            smoothed = moving_avg(y)
            ax1.plot(x[2:-2], smoothed, color=colors[tid], linewidth=2, label=f'Task {tid}')
        else:
            ax1.plot(x, y, color=colors[tid], linewidth=2, label=f'Task {tid}')

ax1.set_xlabel('Episode')
ax1.set_ylabel('Avg Reward')
ax1.set_title('Training Reward per Episode')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_ylim(-0.1, 1.1)

ax2 = axes[1]
tasks       = ['Task 1', 'Task 2', 'Task 3', 'Overall']
keys        = ['task1', 'task2', 'task3', 'overall']
before_vals = [baseline_scores.get(k, 0) for k in keys]
after_vals  = [trained_scores.get(k, 0) for k in keys]

x     = np.arange(len(tasks))
width = 0.35

bars1 = ax2.bar(x - width/2, before_vals, width, label='Before Training', color='#95a5a6')
bars2 = ax2.bar(x + width/2, after_vals,  width, label='After GRPO',      color='#2ecc71')

for bar in bars1:
    ax2.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
             f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax2.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
             f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=9,
             fontweight='bold', color='#27ae60')

ax2.set_xticks(x)
ax2.set_xticklabels(tasks)
ax2.set_ylabel('Score (0-1)')
ax2.set_title('Before vs After GRPO Training')
ax2.legend()
ax2.grid(True, alpha=0.3, axis='y')
ax2.set_ylim(0, 1.15)

plt.tight_layout()
plt.savefig('/content/grpo_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to /content/grpo_results.png')

In [ ]:
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Model saved to {OUTPUT_DIR}')

try:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    api.create_repo(HF_REPO_ID, exist_ok=True, private=False)
    api.upload_folder(folder_path=OUTPUT_DIR, repo_id=HF_REPO_ID, repo_type='model')
    api.upload_file(path_or_fileobj='/content/grpo_results.png', path_in_repo='grpo_results.png', repo_id=HF_REPO_ID, repo_type='model')
    print(f'Model pushed to: https://huggingface.co/{HF_REPO_ID}')
except Exception as e:
    print(f'Push failed: {e}')
    print(f'Model is saved locally at {OUTPUT_DIR}')

In [ ]:
from google.colab import files
files.download('/content/grpo_results.png')

print('\n' + '='*50)
print('FINAL TRAINING SUMMARY')
print('='*50)
print(f'Model:     {MODEL_NAME}')
print(f'Algorithm: GRPO')
print(f'Episodes:  {N_EPISODES}')
print(f'Env:       {ENV_BASE_URL}')
print()
print(f'{"Task":<10} {"Before":>8} {"After":>8} {"Delta":>8}')
print('-' * 38)
for key, label in [("task1","Task 1"),("task2","Task 2"),("task3","Task 3"),("overall","Overall")]:
    b = baseline_scores.get(key, 0)
    a = trained_scores.get(key, 0)
    d = a - b
    print(f'{label:<10} {b:>8.3f} {a:>8.3f} {d:>+8.3f}')
print('='*50)
print(f'Model: https://huggingface.co/{HF_REPO_ID}')